In [6]:
from LLMGraphTransformer import LLMGraphTransformer
from LLMGraphTransformer.schema import NodeSchema, RelationshipSchema
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document


from dotenv import load_dotenv
import os


In [7]:
text_dpa="""
The Data Protection Officer (DPO)
The Data Protection Officer has the role of ensuring that the organisation is processing
personal data in compliance with GDPR rules. It has to be designated on the basis of
professional qualities and knowledge of data protection law and practices. In some in-
stances, the data controller has an obligation to appoint a data protection officer. This
is the case if:
· the processing is carried out by a public authority;
· the core activities of the controller or the processor require “by virtue of their na-
ture, their scope and/or their purposes, regular and systematic monitoring of data
subjects on a large scale” (Art. 37, (1) b); or
· the core activities of the controller or the processor consist of processing, on a
large scale, special categories of data or personal data relating to criminal convic-
tions (see special categories of data).
However, national legislation might specify further cases where there is an obligation
to appoint a DPO. In Germany, for instance, every organisation needs to appoint a DPO
if there are more than 10 people constantly involved with automatic processing of data.
If the DPO is to be a member of staff, then the works council has a right of co-determi-
nation. In general, it is strongly advised to appoint a DPO even if it is not an obligation.
The DPO’s main task is to advise the controller and processors about how to comply
with the regulation. In particular, the DPO’s roles are to:
· inform and advise the employees of the data controller or processor on their obli-
gations arising from the GDPR and any other national data protection rules;
· monitor compliance with the data protection legislation;
· check if the responsibilities of the controller and processor have correctly been as-
signed, and if awareness-raising and sufficient training for staff have taken place;
· provide advice on the data protection impact assessment and monitor its perfor-
mance;
· cooperate with the supervisory authority, and to act as a contact person for them;
and
· be available for inquiries from data subjects (individuals whose data the controller
possesses), for issues of data processing or where individuals want to make use of
one of their rights (these will be discussed later).
Data protection officers can work for several organisations as long as they remain “easi-
ly accessible”. Furthermore, they can be a member of the staff or fulfil their tasks on the
basis of a service contract.
DPOs also enjoy specific rights such as to have sufficient resources to fulfil the tasks
assigned to them. They also have the right of access to the entities’ data processing
personnel and operations and to training in order to “maintain their expert knowledge”.
Moreover, data protection officers should have significant independence in carring out
their tasks and reporting to the highest management level. They can also fulfil other
tasks as long as there is no conflict of interest with their role as DPO. Many of the tasks
that are assigned to the data controller (e.g. documenting processing activities etc.) can
hence also be assumed by the DPO. Lastly, DPOs enjoy a high level of job security. They
cannot be fired, nor can penalties be imposed on the ground of performing their re-
sponsibilities as a DPO. There is no length of tenure for this position.
This has financial and staff implications for public authorities as well as companies
and organisations who process a large amount of data, which may be reduced by ap-
pointing one DPO for several organisations.
"""

In [8]:
node_schemas = [
    NodeSchema(
        "DataProtectionOfficer",
        ["name", "designation_basis", "professional_qualities", "knowledge", "independence", "tenure", "accessibility", "rights"],
        "Represents the DPO ensuring GDPR compliance"
    ),
    NodeSchema(
        "Controller",
        ["name", "core_activities", "obligations", "responsibilities"],
        "Represents the data controller responsible for determining purposes and means of processing"
    ),
    NodeSchema(
        "Processor",
        ["name", "core_activities", "obligations", "responsibilities"],
        "Represents the data processor acting on behalf of the controller"
    ),
    NodeSchema(
        "SupervisoryAuthority",
        ["name", "jurisdiction", "contact_person"],
        "Represents the authority overseeing GDPR compliance"
    ),
    NodeSchema(
        "DataSubject",
        ["rights", "inquiries", "personal_data"],
        "Represents individuals whose personal data is processed"
    ),
    NodeSchema(
        "Organization",
        ["name", "sector", "obligation_to_appoint_dpo", "works_council_rights"],
        "Represents a company, institution, or public authority subject to GDPR rules"
    )
]


In [9]:
relationship_schemas = [
    RelationshipSchema("DataProtectionOfficer", "ADVISES", "Controller", ["scope", "obligation"]),
    RelationshipSchema("DataProtectionOfficer", "ADVISES", "Processor", ["scope", "obligation"]),
    RelationshipSchema("DataProtectionOfficer", "COOPERATES_WITH", "SupervisoryAuthority", ["contact_person", "jurisdiction"]),
    RelationshipSchema("DataProtectionOfficer", "AVAILABLE_FOR", "DataSubject", ["inquiry_type", "rights_invoked"]),
    RelationshipSchema("Controller", "PROCESSES", "DataSubject", ["data_type", "purpose", "scale"]),
    RelationshipSchema("Processor", "PROCESSES_ON_BEHALF_OF", "Controller", ["contract_basis", "scope"]),
    RelationshipSchema("Organization", "APPOINTS", "DataProtectionOfficer", ["obligation_basis", "national_legislation"]),
    RelationshipSchema("Organization", "SUBJECT_TO", "SupervisoryAuthority", ["jurisdiction"]),
    RelationshipSchema("WorksCouncil", "HAS_RIGHT_OF", "Organization", ["co_determination"]),
    RelationshipSchema("DataProtectionOfficer", "REPORTS_TO", "Organization", ["management_level"])
]


In [ ]:
api_key ="<OPENAI_API_KEY>"

model_name = "gpt-4o"

llm = ChatOpenAI(
    api_key=api_key,
    
    model=model_name,
    temperature=0,
)

In [21]:
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=node_schemas,
    allowed_relationships=relationship_schemas,
    additional_instructions=""
)


ETRACTION FOR BIGGER PARAGRAPH

In [23]:
document = Document(page_content=text_dpa)
graph_document = llm_transformer.convert_to_graph_document(document)


In [24]:
for node in graph_document.nodes:
    print(node)
    print("\n")
    

id='Data Protection Officer' type='Dataprotectionofficer' properties={'designation_basis': ['professional qualities', 'knowledge of data protection law and practices'], 'professional_qualities': ['inform and advise employees', 'monitor compliance', 'provide advice on data protection impact assessment', 'cooperate with supervisory authority', 'be available for inquiries'], 'knowledge': ['data protection law', 'practices'], 'independence': ['significant independence', 'reporting to highest management level'], 'accessibility': 'easily accessible', 'rights': ['sufficient resources', 'access to data processing personnel', 'training', 'job security']}


id='Data Controller' type='Controller' properties={'obligations': ['appoint a data protection officer', 'document processing activities'], 'responsibilities': ['assign responsibilities correctly', 'awareness-raising and training for staff']}


id='Data Processor' type='Processor' properties={'obligations': 'appoint a data protection officer',

In [25]:
for rel in graph_document.relationships:
    print(rel)
    print("\n")

source=Node(id='Data Protection Officer', type='Dataprotectionofficer', properties={'designation_basis': ['professional qualities', 'knowledge of data protection law and practices'], 'professional_qualities': ['inform and advise employees', 'monitor compliance', 'provide advice on data protection impact assessment', 'cooperate with supervisory authority', 'be available for inquiries'], 'knowledge': ['data protection law', 'practices'], 'independence': ['significant independence', 'reporting to highest management level'], 'accessibility': 'easily accessible', 'rights': ['sufficient resources', 'access to data processing personnel', 'training', 'job security']}) target=Node(id='Data Controller', type='Controller', properties={'obligations': ['appoint a data protection officer', 'document processing activities'], 'responsibilities': ['assign responsibilities correctly', 'awareness-raising and training for staff']}) type='ADVISES' properties={'scope': 'compliance with regulation'}


source

I have taken the Data Protection Officer excerpt from our text corpus.
To leverage the LLMGraphTransformer module, one has to tediously work on getting Node scheman Relationship right.
Even after 

In [29]:
text_dpa_larger="""

The Data Controller and Data Processor
In general terms, the data controller the natural or legal person (could be a company
or a non-profit organisation), public authority, agency or other body which, alone or
jointly with others, the purposes, conditions and means of processing personal data.
In other words, the controller owns the data and sets the rules how it is to be collected
and processed. The controller therefore keeps a record of all processing activities and
furthermore designates one or more data processors that can, in the name of the data
controller, collect and process the data.
However, this distinction does not always clearly apply in practice, although it has
existed in previous data protection regulations, and the status of employees of data
controllers is still disputed. According to the UK data protection authority an employee
of a data controller cannot be considered as a data processor2, which would suggest
that he or she is a data controller. However, if the same processing activities would be
outsourced (e.g. to an external consultant), this external party would be considered as a
data processor. The GDPR lacks a crucial point in the definition, which has implications
for liability and responsibility.

The Data Protection Officer (DPO)
The Data Protection Officer has the role of ensuring that the organisation is processing
personal data in compliance with GDPR rules. It has to be designated on the basis of
professional qualities and knowledge of data protection law and practices. In some in-
stances, the data controller has an obligation to appoint a data protection officer. This
is the case if:
· the processing is carried out by a public authority;
· the core activities of the controller or the processor require “by virtue of their na-
ture, their scope and/or their purposes, regular and systematic monitoring of data
subjects on a large scale” (Art. 37, (1) b); or
· the core activities of the controller or the processor consist of processing, on a
large scale, special categories of data or personal data relating to criminal convic-
tions (see special categories of data).
However, national legislation might specify further cases where there is an obligation
to appoint a DPO. In Germany, for instance, every organisation needs to appoint a DPO
if there are more than 10 people constantly involved with automatic processing of data.
If the DPO is to be a member of staff, then the works council has a right of co-determi-
nation. In general, it is strongly advised to appoint a DPO even if it is not an obligation.
The DPO’s main task is to advise the controller and processors about how to comply
with the regulation. In particular, the DPO’s roles are to:
· inform and advise the employees of the data controller or processor on their obli-
gations arising from the GDPR and any other national data protection rules;
· monitor compliance with the data protection legislation;
· check if the responsibilities of the controller and processor have correctly been as-
signed, and if awareness-raising and sufficient training for staff have taken place;
· provide advice on the data protection impact assessment and monitor its perfor-
mance;
· cooperate with the supervisory authority, and to act as a contact person for them;
and
· be available for inquiries from data subjects (individuals whose data the controller
possesses), for issues of data processing or where individuals want to make use of
one of their rights (these will be discussed later).
Data protection officers can work for several organisations as long as they remain “easi-
ly accessible”. Furthermore, they can be a member of the staff or fulfil their tasks on the
basis of a service contract.
DPOs also enjoy specific rights such as to have sufficient resources to fulfil the tasks
assigned to them. They also have the right of access to the entities’ data processing
personnel and operations and to training in order to “maintain their expert knowledge”.
Moreover, data protection officers should have significant independence in carring out
their tasks and reporting to the highest management level. They can also fulfil other
tasks as long as there is no conflict of interest with their role as DPO. Many of the tasks
that are assigned to the data controller (e.g. documenting processing activities etc.) can
hence also be assumed by the DPO. Lastly, DPOs enjoy a high level of job security. They
cannot be fired, nor can penalties be imposed on the ground of performing their re-
sponsibilities as a DPO. There is no length of tenure for this position.
This has financial and staff implications for public authorities as well as companies
and organisations who process a large amount of data, which may be reduced by ap-
pointing one DPO for several organisations.

"""

In [30]:
document = Document(page_content=text_dpa_larger)
graph_document = llm_transformer.convert_to_graph_document(document)

In [31]:
for node in graph_document.nodes:
    print(node)
    print("\n")

id='Data Protection Officer' type='Dataprotectionofficer' properties={'professional_qualities': 'knowledge of data protection law and practices', 'independence': 'significant independence in carrying out tasks', 'tenure': 'no length of tenure', 'accessibility': 'easily accessible', 'rights': ['sufficient resources', 'right of access to data processing personnel and operations', 'training to maintain expert knowledge', 'high level of job security']}


id='Data Controller' type='Controller' properties={'obligations': ['appoint a data protection officer', 'keep a record of all processing activities'], 'responsibilities': 'determine purposes, conditions, and means of processing personal data'}


id='Data Processor' type='Processor' properties={'responsibilities': 'collect and process data on behalf of the data controller'}


id='Supervisory Authority' type='Supervisoryauthority' properties={}


id='Data Subject' type='Datasubject' properties={'rights': 'make use of their rights regarding d

In [32]:
for rel in graph_document.relationships:
    print(rel)
    print("\n")

source=Node(id='Data Protection Officer', type='Dataprotectionofficer', properties={'professional_qualities': 'knowledge of data protection law and practices', 'independence': 'significant independence in carrying out tasks', 'tenure': 'no length of tenure', 'accessibility': 'easily accessible', 'rights': ['sufficient resources', 'right of access to data processing personnel and operations', 'training to maintain expert knowledge', 'high level of job security']}) target=Node(id='Data Controller', type='Controller', properties={'obligations': ['appoint a data protection officer', 'keep a record of all processing activities'], 'responsibilities': 'determine purposes, conditions, and means of processing personal data'}) type='ADVISES' properties={}


source=Node(id='Data Protection Officer', type='Dataprotectionofficer', properties={'professional_qualities': 'knowledge of data protection law and practices', 'independence': 'significant independence in carrying out tasks', 'tenure': 'no le